# Column Standardization To-Do List

### CONTEXT 
This is a journal that assists with the sanity checks prior to manually identify relevant samples after the [previous notebook](Stage1.0_getDiseases.ipynb) where we identified the relevant studies for the project.

journal contains standardization of tissues and cell types followed by a merger of the manually currated metadata with the GEO metadata, to create a unified metadata file for downstream analysis. 

Columns of note are : 

```txt
   CLASSIFICATION		- identification of disease or control samples
   SAMPLE_TYPE		        - blood or primary
   CELL_SOURCE		        - cell source identified from GEO metadata
   TISSUE_SOURCE		- tissue source identified from GEO metadata
   CONTEXT			- timepoints or other relevant information that may be useful
```

`labeling/TB_man.tsv` and `labeling/TBmicro_man.tsv` are files with ALL CAPS column names, all caps column names indicated that they are manually curated.
_____ 

output of this notebook are the [RNA unified metadata files](../data/RNAseq_data_forDE/TB_sample_metadata_classification.tsv) and [microarray unified metadata files](../data/microarray_data_forDE/clean_TB_sample_metadata_classification.tsv) that will be used for DESEQ and microarray analysis respectively.

##### Load libraries

In [1]:
import pandas as pd
import os
import re

# Help Functions

##### Column-specific Helper Functions

In [8]:
def print_unique_counts_and_values(df, columns):
    """
    Prints the number of unique values and the unique values themselves for each column in columns.
    """
    print("Unique counts for specified columns:")
    print(df[columns].nunique())
    print("\n")
    for col in columns:
        print(f"Unique values in {col}:")
        print(df[col].unique())
        print()

def remap_and_insert_clean_column(df: pd.DataFrame, column: str, remap_dict: dict, suffix: str):
    """
    Remap values in a column using remap_dict, add a new column with remapped values,
    and reorder columns so the new column is next to the original.
    
    Parameters:
        df (pd.DataFrame): Input dataframe (already cleaned).
        column (str): Column to remap.
        remap_dict (dict): Dictionary for remapping.
        suffix (str): Suffix for new column name (default: '_CLEAN').
    
    Returns:
        pd.DataFrame: DataFrame with new column and reordered columns.
    """
    print(f"Original values in {column}:")
    print(df[column].unique())
    print(df[column].nunique())
    clean_col = f"{column}_{suffix}"
    df[clean_col] = df[column].map(remap_dict).fillna(df[column])
    # Move the new column to the right of the original
    cols = list(df.columns)
    cols.remove(clean_col)
    insert_at = cols.index(column) + 1
    cols = cols[:insert_at] + [clean_col] + cols[insert_at:]
    df = df[cols]
    print(f"Remapped values in {clean_col}:")
    print(df[clean_col].unique())
    print(df[clean_col].nunique())
    return df.head()

## TB RNA-SEQ

In [38]:
TBman = pd.read_csv('../data/labeling/TB_man.tsv', sep='\t') 
TBman.head()

,GSE_ID,GSM_ID,CLASSIFICATION,SAMPLE_TYPE,CELL_SOURCE,TISSUE_SOURCE,CONTEXT
0,GSE84076,GSM2226808,Healthy Control without treatment,blood_sample,whole blood,WHOLE_BLOOD,NaN
1,GSE84076,GSM2226809,Disease without treatment,blood_sample,whole blood,WHOLE_BLOOD,NaN
2,GSE84076,GSM2226817,Disease without treatment,blood_sample,whole blood,WHOLE_BLOOD,NaN
3,GSE84076,GSM2226825,Healthy Control without treatment,blood_sample,whole blood,WHOLE_BLOOD,NaN
4,GSE84076,GSM2226829,Healthy Control without treatment,blood_sample,whole blood,WHOLE_BLOOD,NaN


In [ ]:
print_unique_counts_and_values(TBman, ['CLASSIFICATION', 'SAMPLE_TYPE', 'CELL_SOURCE', 'TISSUE_SOURCE', 'CONTEXT'])

Unique counts for specified columns:
CLASSIFICATION     2
SAMPLE_TYPE        2
CELL_SOURCE       32
TISSUE_SOURCE      9
CONTEXT            6
dtype: int64


Unique values in CLASSIFICATION:
['Healthy Control without treatment' 'Disease without treatment']

Unique values in SAMPLE_TYPE:
['blood_sample' 'primary_sample']

Unique values in CELL_SOURCE:
['whole blood' 'PBMC' 'PBMCs' 'PBMCs (Peripheral Blood Mononuclear Cells)'
 'PBMCs (peripheral blood mononuclear cells)' 'ILC1' 'ILC2' 'ILC3neg'
 'ILC3pos' 'NK' 'susp' 'AM' 'endo' 'gd' 'mait' 'NKT' 'epi'
 'Human stem cell-derived macrophage' 'blood mononuclear cells'
 'CD14+ monocytes' 'PBMC (peripheral blood mononuclear cells)'
 'PBMC (Peripheral Blood Mononuclear Cells)' 'Whole Blood'
 'Bronchoalveolar lavage fluid' 'HepG2' 'Dendritic cells (DCs)'
 'Monocyte-derived dendritic (DCs)' 'blood' 'Blood' 'macrophages'
 'bronchoalveolar lavage' 'macrophage']

Unique values in TISSUE_SOURCE:
['WHOLE_BLOOD' 'PBMC' 'LUNG_TISSUE_GENERAL' 'MACROPHAGE

In [6]:
TB_CELL_SOURCE_REMAP = {
    'whole blood' : 'Blood Cells',
	'Whole Blood' : 'Blood Cells',
	'blood' : 'Blood Cells',
	'Blood' : 'Blood Cells',
	
	'PBMC'  : 'PBMCs',
	'PBMCs' : 'PBMCs',
	'PBMCs (Peripheral Blood Mononuclear Cells)' : 'PBMCs',
    'PBMCs (peripheral blood mononuclear cells)' : 'PBMCs',
	'PBMC (peripheral blood mononuclear cells)' : 'PBMCs',
	'PBMC (Peripheral Blood Mononuclear Cells)' : 'PBMCs',
	
	'blood mononuclear cells' : 'Monocytes',
	
	
	'THP-1' : 'THP1',
	
	'lung' : 'Lung',
	'lung tissue' : 'Lung',
	
	
	'EB7' : 'EB7',
	'CD14+ monocytes' : 'CD14+ Monocytes',
	'HepG2' : 'HepG2',
	
	'Dendritic cells (DCs)' : 'Dendritic Cells',
	'Monocyte-derived dendritic (DCs)' : 'monocyte-derived dendritic cells',
	
	'Human stem cell-derived macrophage' : 'macrophages',
	'macrophages' : 'macrophages',
	'macrophages' : 'macrophages',
	'macrophage' : 'macrophages',

	'bronchoalveolar_lavage' : 'BALF',
	'Bronchoalveolar lavage fluid' : 'BALF',
}

TB_TISSUE_SOURCE_REMAP = {
'WHOLE_BLOOD' : 'Blood',
'BLOOD_GENERAL' : 'Blood',
'PBMC' : 'Blood',
'LUNG_TISSUE_GENERAL' : 'Lung',
'MACROPHAGES_GENERAL' : 'Macrophages',
'CELL_LINE_HEPG2' : 'HepG2',
'MONOCYTES_GENERAL' : 'Monocytes',
'BALF_BRONCHOALVEOLAR_LAVAGE_FLUID' : 'BALF',
'BAL_BRONCHOALVEOLAR_LAVAGE_GENERAL' : 'BALF',
}

In [7]:
remap_and_insert_clean_column(TBman, 'CELL_SOURCE', TB_CELL_SOURCE_REMAP, 'CLEAN')

Original values in CELL_SOURCE:
['whole blood' 'PBMC' 'PBMCs' 'PBMCs (Peripheral Blood Mononuclear Cells)'
 'PBMCs (peripheral blood mononuclear cells)' 'ILC1' 'ILC2' 'ILC3neg'
 'ILC3pos' 'NK' 'susp' 'AM' 'endo' 'gd' 'mait' 'NKT' 'epi'
 'Human stem cell-derived macrophage' 'blood mononuclear cells'
 'CD14+ monocytes' 'PBMC (peripheral blood mononuclear cells)'
 'PBMC (Peripheral Blood Mononuclear Cells)' 'Whole Blood'
 'Bronchoalveolar lavage fluid' 'HepG2' 'Dendritic cells (DCs)'
 'Monocyte-derived dendritic (DCs)' 'blood' 'Blood' 'macrophages'
 'bronchoalveolar lavage' 'macrophage']
32
Remapped values in CELL_SOURCE_CLEAN:
['Blood Cells' 'PBMCs' 'ILC1' 'ILC2' 'ILC3neg' 'ILC3pos' 'NK' 'susp' 'AM'
 'endo' 'gd' 'mait' 'NKT' 'epi' 'macrophages' 'Monocytes'
 'CD14+ Monocytes' 'BALF' 'HepG2' 'Dendritic Cells'
 'monocyte-derived dendritic cells' 'bronchoalveolar lavage']
22


,GSE_ID,GSM_ID,CLASSIFICATION,SAMPLE_TYPE,CELL_SOURCE,CELL_SOURCE_CLEAN,TISSUE_SOURCE,CONTEXT
0,GSE84076,GSM2226808,Healthy Control without treatment,blood_sample,whole blood,Blood Cells,WHOLE_BLOOD,NaN
1,GSE84076,GSM2226809,Disease without treatment,blood_sample,whole blood,Blood Cells,WHOLE_BLOOD,NaN
2,GSE84076,GSM2226817,Disease without treatment,blood_sample,whole blood,Blood Cells,WHOLE_BLOOD,NaN
3,GSE84076,GSM2226825,Healthy Control without treatment,blood_sample,whole blood,Blood Cells,WHOLE_BLOOD,NaN
4,GSE84076,GSM2226829,Healthy Control without treatment,blood_sample,whole blood,Blood Cells,WHOLE_BLOOD,NaN
...,...,...,...,...,...,...,...,...
510,GSE236156,GSM7519008,Disease without treatment,primary_sample,bronchoalveolar lavage,bronchoalveolar lavage,BAL_BRONCHOALVEOLAR_LAVAGE_GENERAL,NaN
511,GSE236156,GSM7519009,Healthy Control without treatment,primary_sample,bronchoalveolar lavage,bronchoalveolar lavage,BAL_BRONCHOALVEOLAR_LAVAGE_GENERAL,NaN
512,GSE236156,GSM7519010,Healthy Control without treatment,primary_sample,bronchoalveolar lavage,bronchoalveolar lavage,BAL_BRONCHOALVEOLAR_LAVAGE_GENERAL,NaN
513,GSE236156,GSM7519011,Disease without treatment,primary_sample,bronchoalveolar lavage,bronchoalveolar lavage,BAL_BRONCHOALVEOLAR_LAVAGE_GENERAL,NaN


In [9]:
remap_and_insert_clean_column(TBman, 'TISSUE_SOURCE', TB_TISSUE_SOURCE_REMAP, 'CLEAN')

Original values in TISSUE_SOURCE:
['WHOLE_BLOOD' 'PBMC' 'LUNG_TISSUE_GENERAL' 'MACROPHAGES_GENERAL'
 'BALF_BRONCHOALVEOLAR_LAVAGE_FLUID' 'CELL_LINE_HEPG2' 'MONOCYTES_GENERAL'
 'BLOOD_GENERAL' 'BAL_BRONCHOALVEOLAR_LAVAGE_GENERAL']
9
Remapped values in TISSUE_SOURCE_CLEAN:
['Blood' 'Lung' 'Macrophages' 'BALF' 'HepG2' 'Monocytes']
6


,GSE_ID,GSM_ID,CLASSIFICATION,SAMPLE_TYPE,CELL_SOURCE,TISSUE_SOURCE,TISSUE_SOURCE_CLEAN,CONTEXT,CELL_SOURCE_CLEAN
0,GSE84076,GSM2226808,Healthy Control without treatment,blood_sample,whole blood,WHOLE_BLOOD,Blood,NaN,Blood Cells
1,GSE84076,GSM2226809,Disease without treatment,blood_sample,whole blood,WHOLE_BLOOD,Blood,NaN,Blood Cells
2,GSE84076,GSM2226817,Disease without treatment,blood_sample,whole blood,WHOLE_BLOOD,Blood,NaN,Blood Cells
3,GSE84076,GSM2226825,Healthy Control without treatment,blood_sample,whole blood,WHOLE_BLOOD,Blood,NaN,Blood Cells
4,GSE84076,GSM2226829,Healthy Control without treatment,blood_sample,whole blood,WHOLE_BLOOD,Blood,NaN,Blood Cells


In [10]:
# Reorder columns
TBman = TBman[['GSE_ID', 'GSM_ID', 'CLASSIFICATION',
       'SAMPLE_TYPE', 'CELL_SOURCE', 'CELL_SOURCE_CLEAN', 'TISSUE_SOURCE',
       'TISSUE_SOURCE_CLEAN', 'CONTEXT']]

TBman.head()

,GSE_ID,GSM_ID,CLASSIFICATION,SAMPLE_TYPE,CELL_SOURCE,CELL_SOURCE_CLEAN,TISSUE_SOURCE,TISSUE_SOURCE_CLEAN,CONTEXT
0,GSE84076,GSM2226808,Healthy Control without treatment,blood_sample,whole blood,Blood Cells,WHOLE_BLOOD,Blood,NaN
1,GSE84076,GSM2226809,Disease without treatment,blood_sample,whole blood,Blood Cells,WHOLE_BLOOD,Blood,NaN
2,GSE84076,GSM2226817,Disease without treatment,blood_sample,whole blood,Blood Cells,WHOLE_BLOOD,Blood,NaN
3,GSE84076,GSM2226825,Healthy Control without treatment,blood_sample,whole blood,Blood Cells,WHOLE_BLOOD,Blood,NaN
4,GSE84076,GSM2226829,Healthy Control without treatment,blood_sample,whole blood,Blood Cells,WHOLE_BLOOD,Blood,NaN


# Checking for Single Cell RNA-seq data

In [11]:
TB_metadata_path = '../data/RNAseq_data_forDE/metadata'

In [12]:
def keyword_search(path, keyword):
	"""
	Searches all .tsv files in a directory for columns where any cell contains the keyword(s).
	Returns a list of (filename, found_columns) for files with matches, or prints a message if no hits.
	"""
	results = []
	for fname in os.listdir(path):
		full_path = os.path.join(path, fname)
		if os.path.isfile(full_path) and fname.endswith('.tsv'):
			df = pd.read_csv(full_path, sep='\t')
			# keyword can be a string or list of strings
			if isinstance(keyword, str):
				keywords = [keyword]
			else:
				keywords = keyword
			# Find columns where any cell contains any of the keywords (case-insensitive)
			mask = df.apply(lambda col: col.astype(str).str.contains('|'.join(keywords), case=False, na=False))
			columns_with_keyword = mask.any(axis=0)
			found_columns = columns_with_keyword[columns_with_keyword].index.tolist()
			if found_columns:
				results.append((fname, found_columns))
	if not results:
		print(f"No columns containing keyword(s) found in any .tsv file in {path}")
		return None
	return results

In [13]:
keyword_search(TB_metadata_path, ["single-cell", "single cell", "scRNA", "scRNA-seq", "scRNAseq", "scRNA seq"])

No columns containing keyword(s) found in any .tsv file in ../data/RNAseq_data_forDE/metadata


# CONCAT GEO metadata with manual labeling

In [14]:
def extract_cols_with_counts(path):
	"""
	Extracts all unique column names from files in a directory and counts in how many files each column appears.
	Returns a dictionary {column_name: count}.
	"""
	col_counts = {}
	for filename in os.listdir(path):
		file_path = os.path.join(path, filename)
		if os.path.isfile(file_path):
			df = pd.read_csv(file_path, sep='\t')
			for col in set(df.columns):
				col_counts[col] = col_counts.get(col, 0) + 1
	return col_counts

In [15]:
# print the number of files in the path
rna_path = '../data/RNAseq_data_forDE/metadata'
print("Number of files in the path:", len(os.listdir(rna_path)))

# Usage
col_counts = extract_cols_with_counts(rna_path)
# To get sorted list of columns and their counts:
sorted_col_counts = sorted(col_counts.items(), key=lambda x: x[1], reverse=True)
sorted_col_counts

Number of files in the path: 22


[('title', 22),
 ('data_processing.2', 22),
 ('geo_accession', 22),
 ('submission_date', 22),
 ('data_processing', 22),
 ('contact_country', 22),
 ('organism_ch1', 22),
 ('contact_name', 22),
 ('last_update_date', 22),
 ('extract_protocol_ch1', 22),
 ('data_processing.1', 22),
 ('taxid_ch1', 22),
 ('status', 22),
 ('library_source', 22),
 ('contact_address', 22),
 ('contact_zip/postal_code', 22),
 ('molecule_ch1', 22),
 ('relation.1', 22),
 ('library_strategy', 22),
 ('platform_id', 22),
 ('data_row_count', 22),
 ('supplementary_file_1', 22),
 ('characteristics_ch1', 22),
 ('contact_city', 22),
 ('contact_institute', 22),
 ('channel_count', 22),
 ('source_name_ch1', 22),
 ('extract_protocol_ch1.1', 22),
 ('series_id', 22),
 ('library_selection', 22),
 ('instrument_model', 22),
 ('type', 22),
 ('relation', 22),
 ('characteristics_ch1.1', 21),
 ('contact_email', 19),
 ('treatment_protocol_ch1', 18),
 ('growth_protocol_ch1', 17),
 ('characteristics_ch1.2', 17),
 ('data_processing.3', 17),

In [16]:
# list of the most common and relevant columns for differentiation
cols_of_interest = [
	'series_id',
	'geo_accession',
	'title',
	'source_name_ch1',
	'organism_ch1',
	'characteristics_ch1',
	'characteristics_ch1.1',
	'characteristics_ch1.2',
	'molecule_ch1',
	'platform_id',
	'library_source',
	'library_strategy',
	'cell type:ch1',
	'tissue:ch1',
	'time:ch1',
	'time point:ch1',
]

In [17]:
def create_mega_df_from_dir(input_dir, cleandf, cols_of_interest, use_NA_string=False):
    """
    Iterates over all files in input_dir, extracts cols_of_interest, 
    keeps only rows with series_id and geo_accession in cleandf,
    and returns the combined DataFrame. 
    Missing columns are filled with 'NA' if use_NA_string is True, otherwise left as NaN.
    """
    mega_df = []
    valid_gse = set(cleandf['GSE_ID'])
    valid_gsm = set(cleandf['GSM_ID'])

    for fname in os.listdir(input_dir):
        if fname.endswith('.tsv'):
            fpath = os.path.join(input_dir, fname)
            df = pd.read_csv(fpath, sep='\t')
            # Ensure all columns of interest are present, fill missing with NaN
            df = df.reindex(columns=cols_of_interest)
            # Filter rows
            if 'series_id' in df.columns and 'geo_accession' in df.columns:
                df = df[df['series_id'].isin(valid_gse) & df['geo_accession'].isin(valid_gsm)]
                mega_df.append(df)
    if mega_df:
        mega_df = pd.concat(mega_df, ignore_index=True)
        if use_NA_string:
            mega_df = mega_df.fillna('NA')
        return mega_df
    else:
        print("No data found matching criteria.")
        return pd.DataFrame(columns=cols_of_interest)

In [18]:
TBmeta = create_mega_df_from_dir(rna_path, TBman, cols_of_interest)

In [24]:
TBmeta.columns

Index(['series_id', 'geo_accession', 'title', 'source_name_ch1',
       'organism_ch1', 'characteristics_ch1', 'characteristics_ch1.1',
       'characteristics_ch1.2', 'molecule_ch1', 'platform_id',
       'library_source', 'library_strategy', 'cell type:ch1', 'tissue:ch1',
       'time:ch1', 'time point:ch1'],
      dtype='object')

##### merge metadata with signature data

In [25]:
columns_list = [
	'organism_ch1',
	'GSE_ID', 
	'GSM_ID',
	'series_id',
	'geo_accession',
	'classification',
	'title',
	'sample_type',
	'tissue_cell',
	'imputed_tissue',
	'source_name_ch1',
	'characteristics_ch1',
	'characteristics_ch1.1',
	'characteristics_ch1.2',
	'molecule_ch1',
	'platform_id',
	'library_source',
	'library_strategy',
	'cell type:ch1',
	'tissue:ch1',
	'time:ch1', 
	'time point:ch1'
	'reason',
	'treatment'
]


In [26]:
def merge_signature_and_metadata(signature_df, metadata_df):
	"""
	Merge signature and metadata dataframes on series_id/GSE_ID and geo_accession/GSM_ID.
	Ensures the resulting columns are ordered according to columns_list.
	"""
	merged = pd.merge(
		signature_df,
		metadata_df,
		left_on=['GSE_ID', 'GSM_ID'],
		right_on=['series_id', 'geo_accession'],
		how='inner'
	)
	# Only keep columns present in columns_list and order them
	merged = merged[[col for col in columns_list if col in merged.columns]]
	return merged

In [ ]:
tbfinal = merge_signature_and_metadata(TBman, TBmeta)

# reorder columns according to columns_list, keeping only those that are present

In [32]:
print(tbfinal.columns)
tbfinal.shape

Index(['organism_ch1', 'GSE_ID', 'GSM_ID', 'series_id', 'geo_accession',
       'title', 'source_name_ch1', 'characteristics_ch1',
       'characteristics_ch1.1', 'characteristics_ch1.2', 'molecule_ch1',
       'platform_id', 'library_source', 'library_strategy', 'cell type:ch1',
       'tissue:ch1', 'time:ch1'],
      dtype='object')


(515, 17)

In [36]:
tbfinal.head()

,organism_ch1,GSE_ID,GSM_ID,series_id,geo_accession,title,source_name_ch1,characteristics_ch1,characteristics_ch1.1,characteristics_ch1.2,molecule_ch1,platform_id,library_source,library_strategy,cell type:ch1,tissue:ch1,time:ch1
0,Homo sapiens,GSE84076,GSM2226808,GSE84076,GSM2226808,Ctrl_Unvac_3,Whole blood,clinical information: Control - BCG - Unvaccin...,tissue: whole blood,NaN,polyA RNA,GPL16791,transcriptomic,RNA-Seq,NaN,whole blood,NaN
1,Homo sapiens,GSE84076,GSM2226809,GSE84076,GSM2226809,ATB_5,Whole blood,clinical information: Active Tuberculosis,tissue: whole blood,NaN,polyA RNA,GPL16791,transcriptomic,RNA-Seq,NaN,whole blood,NaN
2,Homo sapiens,GSE84076,GSM2226817,GSE84076,GSM2226817,ATB_4,Whole blood,clinical information: Active Tuberculosis,tissue: whole blood,NaN,polyA RNA,GPL16791,transcriptomic,RNA-Seq,NaN,whole blood,NaN
3,Homo sapiens,GSE84076,GSM2226825,GSE84076,GSM2226825,Ctrl_Unvac_2,Whole blood,clinical information: Control - BCG - Unvaccin...,tissue: whole blood,NaN,polyA RNA,GPL16791,transcriptomic,RNA-Seq,NaN,whole blood,NaN
4,Homo sapiens,GSE84076,GSM2226829,GSE84076,GSM2226829,Ctrl_Unvac_1,Whole blood,clinical information: Control - BCG - Unvaccin...,tissue: whole blood,NaN,polyA RNA,GPL16791,transcriptomic,RNA-Seq,NaN,whole blood,NaN


## TB micro

In [44]:
# print the number of files in the path
micropath = '../data/microarray_data_forDE/metadata'
print("Number of files in the path:", len(os.listdir(micropath)))

# Usage
col_counts = extract_cols_with_counts(micropath)
# To get sorted list of columns and their counts:
sorted_col_counts = sorted(col_counts.items(), key=lambda x: x[1], reverse=True)
sorted_col_counts

Number of files in the path: 10


[('contact_city', 10),
 ('source_name_ch1', 10),
 ('characteristics_ch1_ethnicity', 10),
 ('refinebio_processor_id', 10),
 ('contact_phone', 10),
 ('contact_country', 10),
 ('hyb_protocol', 10),
 ('contact_name', 10),
 ('description', 10),
 ('characteristics_ch1_time', 10),
 ('characteristics_ch1_smear of index case', 10),
 ('series_id', 10),
 ('characteristics_ch1_sputum_culture', 10),
 ('refinebio_treatment', 10),
 ('platform_id', 10),
 ('characteristics_ch1_healthy control', 10),
 ('refinebio_source_database', 10),
 ('refinebio_cell_line', 10),
 ('characteristics_ch1_infection status', 10),
 ('relation', 10),
 ('characteristics_ch1_region of birth', 10),
 ('contact_state', 10),
 ('characteristics_ch1_treatment', 10),
 ('refinebio_processor_name', 10),
 ('label_ch1', 10),
 ('refinebio_genetic_information', 10),
 ('last_update_date', 10),
 ('type', 10),
 ('characteristics_ch1_geographical region', 10),
 ('growth_protocol_ch1', 10),
 ('refinebio_sex', 10),
 ('characteristics_ch1_diseas

In [48]:
columns_listmicro = [
	'organism_ch1',
	'GSE_ID', 
	'GSM_ID',
	'series_id',
	'geo_accession',
	'CLASSIFICATION',
	'title',
	'SAMPLE_TYPE',
	'CELL',
	'TISSUE_CELL TYPE',
	'TB_TYPE',
	'source_name_ch1',
	'characteristics_ch1',
	'characteristics_ch1.1',
	'characteristics_ch1.2',
	'molecule_ch1',
	'platform_id',
	'cell type:ch1',
	'tissue:ch1',
	'time:ch1',
	'TIMEPOINT'
]

cols_of_interestMICRO = [
	'series_id',
	'geo_accession',
	'title',
	'source_name_ch1',
	'organism_ch1',
	'characteristics_ch1',
	'characteristics_ch1.1',
	'characteristics_ch1.2',
	'molecule_ch1',
	'platform_id',
	'cell type:ch1',
	'tissue:ch1',
	'time:ch1',
]

In [46]:
def merge_signature_and_metadata(signature_df, metadata_df):
	"""
	Merge signature and metadata dataframes on series_id/GSE_ID and geo_accession/GSM_ID.
	Ensures the resulting columns are ordered according to columns_list.
	"""
	merged = pd.merge(
		signature_df,
		metadata_df,
		left_on=['GSE_ID', 'GSM_ID'],
		right_on=['series_id', 'geo_accession'],
		how='inner'
	)
	# Only keep columns present in columns_list and order them
	merged = merged[[col for col in columns_listmicro if col in merged.columns]]
	return merged

In [55]:
cleaned_df_tbmicro = pd.read_csv('../data/labeling/TBmicro_man.tsv', sep='\t')

In [65]:
cleaned_df_tbmicro

,GSE_ID,GSM_ID,CLASSIFICATION,SAMPLE_TYPE,CELL,TISSUE_CELL TYPE,TB_TYPE,TIMEPOINT
0,GSE16250,GSM409134,disease without treatment,PRIMARY_SAMPLE,PMBC,BLOOD,MTB,NaN
1,GSE16250,GSM409135,disease without treatment,PRIMARY_SAMPLE,PMBC,BLOOD,MTB,NaN
2,GSE16250,GSM409136,disease without treatment,PRIMARY_SAMPLE,PMBC,BLOOD,MTB,NaN
3,GSE16250,GSM409137,healthy control without treatment,PRIMARY_SAMPLE,PMBC,BLOOD,NaN,NaN
4,GSE16250,GSM409138,healthy control without treatment,PRIMARY_SAMPLE,PMBC,BLOOD,NaN,NaN
...,...,...,...,...,...,...,...,...
1289,GSE139871,GSM4148009,disease without treatment,PRIMARY_SAMPLE,Monocyte,BLOOD,MTB,NaN
1290,GSE139871,GSM4148010,disease without treatment,PRIMARY_SAMPLE,Monocyte,BLOOD,MTB,NaN
1291,GSE139871,GSM4148011,disease without treatment,PRIMARY_SAMPLE,Monocyte,BLOOD,MTB,NaN
1292,GSE139871,GSM4148012,disease without treatment,PRIMARY_SAMPLE,Monocyte,BLOOD,MTB,NaN


In [ ]:
TBmicrometa= create_mega_df_from_dir(micropath, cleaned_df_tbmicro, cols_of_interestMICRO)

In [58]:
microfinal = merge_signature_and_metadata(cleaned_df_tbmicro, TBmicrometa)

In [ ]:
microfinal.head 

<bound method NDFrame.head of      organism_ch1     GSE_ID      GSM_ID  series_id geo_accession  \
0    Homo sapiens   GSE16250   GSM409134   GSE16250     GSM409134   
1    Homo sapiens   GSE16250   GSM409135   GSE16250     GSM409135   
2    Homo sapiens   GSE16250   GSM409136   GSE16250     GSM409136   
3    Homo sapiens   GSE16250   GSM409137   GSE16250     GSM409137   
4    Homo sapiens   GSE16250   GSM409138   GSE16250     GSM409138   
..            ...        ...         ...        ...           ...   
919  Homo sapiens  GSE139871  GSM4148009  GSE139871    GSM4148009   
920  Homo sapiens  GSE139871  GSM4148010  GSE139871    GSM4148010   
921  Homo sapiens  GSE139871  GSM4148011  GSE139871    GSM4148011   
922  Homo sapiens  GSE139871  GSM4148012  GSE139871    GSM4148012   
923  Homo sapiens  GSE139871  GSM4148013  GSE139871    GSM4148013   

                        CLASSIFICATION                        title  \
0            disease without treatment                      H37Ra-1   

In [61]:
# drop gse_id and gsm_id columns from tbfinal2
microfinal = microfinal.drop(columns=['GSE_ID', 'GSM_ID'])

file currently reflects a dimension row count of 724 there were additional changes that were made to the file downstream